# FinGPT-Forecaster (aus FinRobots Tutorial `agent_fingpt_forecaster.ipynb`)

Der Market Analyst holt Profil, News und Kennzahlen (Finnhub) und Kursdaten (Yahoo), nennt Chancen und Risiken und sagt die Kursbewegung der naechsten Woche voraus. Jede Vorhersage wird protokolliert (`forecasts/forecast_log.csv`) und kann nach einer Woche mit `scripts/score_forecasts.py` ausgewertet werden.

In [ ]:
# 1) Pfade und Imports
import os, sys, json, glob
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
FR = ROOT / "finrobot_autogen"
sys.path.insert(0, str(AIF))

import autogen
from finrobot.utils import register_keys_from_json
from lib.forecaster import forecast


In [ ]:
# 2) Modell und Schluessel
MODEL = "anthropic/claude-haiku-5.5"   # z. B. "google/gemini-4-neon"

base = json.load(open(FR / "OAI_CONFIG_LIST", encoding="utf-8"))
entry = next((e for e in base if e["model"] == MODEL), None) or dict(base[0], model=MODEL)  # gleicher OpenRouter-Key
llm_config = {"config_list": [entry], "timeout": 300, "temperature": 0.3,
              "parallel_tool_calls": False, "max_tokens": 8000}
if MODEL.startswith("google/"):
    llm_config["extra_body"] = {"reasoning": {"effort": "low"}}
register_keys_from_json(str(FR / "config_api_keys"))


In [ ]:
# 3) Ticker (einer oder mehrere); extended=True gibt zusaetzlich Konsens, Revisionen und Quality Scores
TICKERS = ["RKLB"]
EXTENDED = False


In [ ]:
# 4) Vorhersagen
for t in TICKERS:
    answer, pred = forecast(llm_config, t, model_name=MODEL, extended=EXTENDED)


In [ ]:
# 5) Protokoll anzeigen
import pandas as pd
pd.read_csv(AIF / "forecasts" / "forecast_log.csv").tail(10)
